# Vapor Pressure Data Preparation

## Notebook Contract
- Primary input: `./Datasets/Raw_Dataset/YawsVP_in-house_digitized.csv` (in-house digitization; authoritative Antoine coefficients, ranges, and codes, with locally recovered SMILES)
- Adapter behavior: keep table 1, use its embedded SMILES directly, rename the in-house columns to the established pipeline schema, and remove review/provenance columns before the existing cleaning logic.
- No PubChem or other network lookup is performed.
- Required adapted columns: `N`, `CAS`, `NAME`, `FORMULA`, `A`, `B`, `C`, `TMIN`, `TMAX`, `code`, `smiles`.


## 1. Setup and Load Raw Data


In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from rdkit import Chem
from sklearn.model_selection import train_test_split

# Configuration
MAIN_RAW_PATH = './Datasets/Raw_Dataset/YawsVP_in-house_digitized.csv'
CURATED_PATH = './Datasets/Curated Dataset(s)/complete_organic_table_with_smiles.csv'
CLEAN_DIR = './Datasets/Curated Dataset(s)'
TRAINABLE_DIR = './Datasets/trainable_data'
RANDOM_SEED = 42
MAX_HEAVY_ATOMS = 40

# Source-specific exclusions are omitted from this publication draft.
EXCLUDE_SMILES = set()

# Load the in-house table and adapt it to the narrow schema used downstream.
main_source_df = pd.read_csv(MAIN_RAW_PATH, dtype=str, keep_default_na=False)

required_source_cols = {
    'table_no', 'no', 'formula', 'name', 'cas_no', 'smiles',
    'A', 'B', 'C', 'tmin_c', 'tmax_c', 'code'
}
missing_source_cols = required_source_cols - set(main_source_df.columns)
if missing_source_cols:
    raise ValueError(f'Missing required columns in in-house vapor-pressure file: {sorted(missing_source_cols)}')

main_df = main_source_df.loc[
    main_source_df['table_no'].astype(str).str.strip().eq('1'),
    ['no', 'formula', 'name', 'cas_no', 'smiles', 'A', 'B', 'C', 'tmin_c', 'tmax_c', 'code']
].rename(columns={
    'no': 'N', 'formula': 'FORMULA', 'name': 'NAME', 'cas_no': 'CAS',
    'tmin_c': 'TMIN', 'tmax_c': 'TMAX',
})
main_df['CAS'] = main_df['CAS'].astype(str).str.strip().replace('', '---')

print(f"Loaded in-house organic rows: {len(main_df)}")
print(f"Rows with embedded SMILES: {main_df['smiles'].astype(str).str.strip().ne('').sum()}")


## 2. Merge Known SMILES and Fill Missing Entries


In [ ]:
def canonicalize_smiles(smiles):
    if pd.isna(smiles) or str(smiles).strip() == '':
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None


# The locally recovered values are embedded in the in-house file, so no auxiliary
# lookup or network recovery is needed during normal data preparation.
merged_df = main_df.copy()
merged_df['smiles'] = merged_df['smiles'].apply(canonicalize_smiles)
merged_df = merged_df.dropna(subset=['smiles']).copy()
merged_df = merged_df[merged_df['smiles'].astype(str).str.strip() != '']
merged_df = merged_df.drop_duplicates(subset=['CAS', 'NAME', 'FORMULA'], keep='first')

os.makedirs(os.path.dirname(CURATED_PATH), exist_ok=True)
merged_df.to_csv(CURATED_PATH, index=False)

coverage = merged_df['smiles'].notna().sum() / len(main_df)
print(f'Curated rows with valid SMILES: {len(merged_df)}')
print(f'SMILES coverage vs in-house organic table: {coverage:.1%}')
print(f'Saved curated dataset: {CURATED_PATH}')


## 3. Filter to Vapor Pressure Training Scope and Clean Molecules


In [ ]:
def clean_smiles_dataset(df, output_path, threshold=0.01, max_heavy_atoms=40):
    smiles_col = None
    for candidate in ['SMILES', 'smiles']:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")

    def canonicalize_smiles_local(smiles):
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            return None

    exclude_raw = set(globals().get('EXCLUDE_SMILES', set()))
    exclude_set = set()
    for s in exclude_raw:
        can = canonicalize_smiles_local(s)
        if can is not None:
            exclude_set.add(can)
        exclude_set.add(str(s))

    def is_organic(mol):
        return any(atom.GetSymbol() == 'C' for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = {
            3,4,11,12,13,14,19,20,21,22,23,24,25,26,27,28,29,30,
            31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        }
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def has_isotopic_label(mol):
        # Remove explicitly isotopically labeled compounds such as deuterated SMILES ([2H]).
        return any(atom.GetIsotope() > 0 for atom in mol.GetAtoms())


    def count_heavy_atoms(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.nan
        return mol.GetNumHeavyAtoms()

    def analyze_elements(df_local, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            'Total_Atom_Count': pd.Series(atom_type_counts),
            'Molecule_Count': pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df_local, smiles_column, threshold_local):
        atom_summary_local = analyze_elements(df_local, smiles_column=smiles_column)
        total_molecules = len(df_local)
        if total_molecules == 0:
            return df_local.copy(), atom_summary_local, [], 0

        rare_elements = atom_summary_local[
            atom_summary_local['Molecule_Count'] / total_molecules < threshold_local
        ].index.tolist()

        mask = []
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))

        filtered_df = df_local[mask].reset_index(drop=True)
        removed_count = len(df_local) - len(filtered_df)
        return filtered_df, atom_summary_local, rare_elements, removed_count

    stats = {'invalid': [], 'multi_component': [], 'excluded_smiles': [], 'inorganic': [], 'organometallic': [], 'ion': [], 'isotopic_label': [], 'kept': []}

    for _, row in df.iterrows():
        row_copy = row.copy()
        smi = row_copy[smiles_col]
        cano = canonicalize_smiles_local(smi)
        if cano is None:
            stats['invalid'].append(row_copy)
            continue

        # Filter out multi-component SMILES (e.g., salts/mixtures like 'CCO.Cl')
        if '.' in str(cano):
            row_copy[smiles_col] = cano
            stats['multi_component'].append(row_copy)
            continue

        # Filter out known-bad SMILES (canonicalized)
        if exclude_set and cano in exclude_set:
            row_copy[smiles_col] = cano
            stats['excluded_smiles'].append(row_copy)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats['invalid'].append(row_copy)
            continue

        row_copy[smiles_col] = cano

        if not is_organic(mol):
            stats['inorganic'].append(row_copy)
            continue
        if is_organometallic(mol):
            stats['organometallic'].append(row_copy)
            continue
        if is_ion(mol):
            stats['ion'].append(row_copy)
            continue
        if has_isotopic_label(mol):
            stats['isotopic_label'].append(row_copy)
            continue

        stats['kept'].append(row_copy)

    pre_dedup_df = pd.DataFrame(stats['kept'])
    pre_dedup_count = len(pre_dedup_df)
    dedup_df = pre_dedup_df.drop_duplicates(subset=[smiles_col], keep='first')
    dedup_removed = pre_dedup_count - len(dedup_df)

    cleaned_df, atom_summary, rare_elements, rare_removed = remove_rare_elements(dedup_df, smiles_col, threshold)

    cleaned_df['heavy_atoms'] = cleaned_df[smiles_col].apply(count_heavy_atoms)
    heavy_before = len(cleaned_df)
    cleaned_df = cleaned_df[cleaned_df['heavy_atoms'] <= max_heavy_atoms].copy()
    heavy_removed = heavy_before - len(cleaned_df)

    import inspect
    frame = inspect.currentframe().f_back
    df_name = None
    for var_name, var_val in frame.f_locals.items():
        if var_val is df:
            df_name = var_name
            break
    if df_name is None:
        df_name = 'dataset'

    os.makedirs(output_path, exist_ok=True)
    output_file = os.path.join(output_path, f'{df_name}_clean.csv')
    cleaned_df.to_csv(output_file, index=False)

    print(f'Cleaned dataset saved to {output_file}')
    print('\nFilter breakdown (stepwise):')
    running = len(df)

    def print_step(label, removed):
        nonlocal running
        after = running - removed
        print(f'- {label}: removed {removed} | before {running} -> after {after}')
        running = after

    print_step('Invalid SMILES', len(stats['invalid']))
    print_step('Multi-component SMILES (contains .)', len(stats['multi_component']))
    print_step('Excluded SMILES (known-bad)', len(stats['excluded_smiles']))
    print_step('Inorganic (no carbon)', len(stats['inorganic']))
    print_step('Organometallic', len(stats['organometallic']))
    print_step('Ions (non-zero formal charge)', len(stats['ion']))
    print_step('Isotopically labeled SMILES', len(stats['isotopic_label']))
    print_step('Duplicate SMILES', dedup_removed)
    print_step(f'Rare elements (<{threshold*100:.1f}% molecule frequency)', rare_removed)
    print_step(f'Heavy atom count > {max_heavy_atoms}', heavy_removed)

    if rare_elements:
        print(f'Rare elements identified: {rare_elements}')
    else:
        print('Rare elements identified: none')

    print(f'Final retained rows: {len(cleaned_df)}')

    return cleaned_df, atom_summary


# Load curated dataset and keep code 1/2 entries only
raw_data_with_smiles = pd.read_csv(CURATED_PATH)
SMILES_subset_c12 = raw_data_with_smiles[
    raw_data_with_smiles['code'].astype(str).str.contains('1') |
    raw_data_with_smiles['code'].astype(str).str.contains('2')
].copy()

# Standardize SMILES column naming expected by later cells
if 'smiles' in SMILES_subset_c12.columns:
    SMILES_subset_c12 = SMILES_subset_c12.rename(columns={'smiles': 'SMILES'})

print(f'Before code filter: {len(raw_data_with_smiles)}')
print(f'After code=1/2 filter: {len(SMILES_subset_c12)}')

SMILES_subset_c12_clean, atom_summary = clean_smiles_dataset(
    SMILES_subset_c12,
    output_path=CLEAN_DIR,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
)
SMILES_subset_c12_clean_filtered = SMILES_subset_c12_clean.copy()


In [ ]:
# Heavy atom filtering now happens inside the main cleaning cell.
SMILES_subset_c12_clean_filtered = SMILES_subset_c12_clean.copy()


## 4. Generate Temperature-Conditioned Vapor Pressure Samples


In [ ]:
def generate_vapor_pressures_augmented(df, num_points=15, seed=None, verbose=True):
    if seed is not None:
        np.random.seed(seed)

    expanded_rows = []
    logP_min = -6
    logP_max = 3

    for idx, row in df.iterrows():
        A, B, C = row['A'], row['B'], row['C']

        denom1 = A - logP_max
        denom2 = A - logP_min
        if abs(denom1) < 1e-8 or abs(denom2) < 1e-8:
            if verbose:
                print(f'Row {idx} skipped: near-zero denominator.')
            continue

        T1 = B / denom1 - C
        T2 = B / denom2 - C
        if not (np.isfinite(T1) and np.isfinite(T2)):
            if verbose:
                print(f'Row {idx} skipped: non-finite temps.')
            continue

        T_low, T_high = sorted([T1, T2])
        if (T_high - T_low) > 1e6:
            if verbose:
                print(f'Row {idx} skipped: absurd range {T_low} to {T_high}.')
            continue

        T_range = np.random.uniform(low=T_low, high=T_high, size=num_points)

        for T_C in T_range:
            row_dict = row.to_dict()
            row_dict['T_C'] = T_C
            row_dict['log10_P'] = A - B / (T_C + C)
            expanded_rows.append(row_dict)

    return pd.DataFrame(expanded_rows)


train_df, temp_df = train_test_split(
    SMILES_subset_c12_clean_filtered,
    test_size=0.2,
    random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_SEED)

train_expanded = generate_vapor_pressures_augmented(train_df, num_points=15, seed=RANDOM_SEED, verbose=False)
val_expanded = generate_vapor_pressures_augmented(val_df, num_points=15, seed=RANDOM_SEED, verbose=False)
test_expanded = generate_vapor_pressures_augmented(test_df, num_points=15, seed=RANDOM_SEED, verbose=False)

# Keep test regime from existing workflow
test_expanded = test_expanded[test_expanded['log10_P'] >= -4].reset_index(drop=True)

# Downsample train by log10_P bins for a more uniform training distribution
n_bins = 50
samples_per_bin = 2000
train_expanded['log10_P_bin'] = pd.cut(train_expanded['log10_P'], bins=n_bins)

downsampled_frames = []
for _, group in train_expanded.groupby('log10_P_bin', observed=False):
    if len(group) > samples_per_bin:
        downsampled_frames.append(group.sample(samples_per_bin, random_state=RANDOM_SEED))
    else:
        downsampled_frames.append(group)

train_expanded = pd.concat(downsampled_frames, ignore_index=True)
train_expanded = train_expanded.drop(columns=['log10_P_bin'])

train_expanded['split'] = 'train'
val_expanded['split'] = 'val'
test_expanded['split'] = 'test'

print(f'Split sizes after expansion/downsampling -> train: {len(train_expanded)}, val: {len(val_expanded)}, test: {len(test_expanded)}')


## 5. Export Chemprop Training and Feature Files


In [ ]:
os.makedirs(TRAINABLE_DIR, exist_ok=True)

# Labels file for Chemprop
target_df = pd.concat([
    train_expanded[['SMILES', 'log10_P', 'split']],
    val_expanded[['SMILES', 'log10_P', 'split']],
    test_expanded[['SMILES', 'log10_P', 'split']]
], ignore_index=True)
train_csv_path = os.path.join(TRAINABLE_DIR, 'VP_chemprop_train_file.csv')
target_df.to_csv(train_csv_path, index=False)

# Temperature feature export (CSV + NPZ)
feature_df = pd.concat([
    train_expanded[['SMILES', 'T_C', 'split']],
    val_expanded[['SMILES', 'T_C', 'split']],
    test_expanded[['SMILES', 'T_C', 'split']]
], ignore_index=True)
feat_csv_path = os.path.join(TRAINABLE_DIR, 'VP_chemprop_feat_file.csv')
feature_df.to_csv(feat_csv_path, index=False)

T_C_array = feature_df['T_C'].to_numpy().reshape(-1, 1)
feat_npz_path = os.path.join(TRAINABLE_DIR, 'VP_chemprop_feat_file.npz')
np.savez(feat_npz_path, T_C_array)

print(f'Saved training labels: {train_csv_path}')
print(f'Saved temperature features CSV: {feat_csv_path}')
print(f'Saved temperature features NPZ: {feat_npz_path}')


## 6. Final Summary


In [ ]:
print('Run summary:')
print(f'- Raw main rows loaded: {len(main_df)}')
# print(f'- Curated rows with valid SMILES: {len(merged_df)}')
print(f'- Code 1/2 rows before cleaning: {len(SMILES_subset_c12)}')
print(f'- Rows after cleaning: {len(SMILES_subset_c12_clean)}')
print(f'- Total rows exported (labels): {len(target_df)}')
print(f'- Total rows exported (features): {len(feature_df)}')
print(f'- Curated CSV: {CURATED_PATH}')
print(f'- Train CSV: {train_csv_path}')
print(f'- Feature CSV: {feat_csv_path}')
print(f'- Feature NPZ: {feat_npz_path}')
